In [ ]:
import random
from itertools import product

# =====================================================
# Step 1: sentences
# =====================================================
def Sym(n):    return ('sym', n)
def Not(a):    return ('not', a)
def And(a, b): return ('and', a, b)
def Or(a, b):  return ('or', a, b)
def Imp(a, b): return ('imp', a, b)
def Iff(a, b): return ('iff', a, b)


def symbols(s):
    if s[0] == 'sym':
        return {s[1]}
    result = set()
    for sub in s[1:]:
        result |= symbols(sub)
    return result


def to_str(s):
    op = s[0]
    if op == 'sym':
        return s[1]
    if op == 'not':
        return '~' + to_str(s[1])
    sign = {'and': '&', 'or': '|', 'imp': '=>', 'iff': '<=>'}[op]
    return f"({to_str(s[1])} {sign} {to_str(s[2])})"


# =====================================================
# Step 2: semantics
# =====================================================
def pl_true(s, model):
    op = s[0]
    if op == 'sym':
        return model[s[1]]
    if op == 'not':
        return not pl_true(s[1], model)
    a = pl_true(s[1], model)
    b = pl_true(s[2], model)
    if op == 'and': return a and b
    if op == 'or':  return a or b
    if op == 'imp': return (not a) or b
    if op == 'iff': return a == b


# =====================================================
# Step 3: truth table, entailment
# =====================================================
def all_models(syms):
    syms = sorted(syms)
    for values in product([True, False], repeat=len(syms)):
        yield dict(zip(syms, values))


def tt_entails(kb, alpha, limit=200_000):
    """Returns (result, models checked). result is None if limit exceeded."""
    syms = symbols(kb) | symbols(alpha)
    count = 0
    for model in all_models(syms):
        count += 1
        if count > limit:
            return None, count
        if pl_true(kb, model) and not pl_true(alpha, model):
            return False, count
    return True, count


def is_valid(s):
    return all(pl_true(s, m) for m in all_models(symbols(s)))


def is_satisfiable(s):
    return any(pl_true(s, m) for m in all_models(symbols(s)))


def print_truth_table(s):
    syms = sorted(symbols(s))
    print(" ".join(syms), "|", to_str(s))
    for m in all_models(syms):
        row = " ".join("T" if m[x] else "F" for x in syms)
        print(row, "|", "T" if pl_true(s, m) else "F")


# =====================================================
# Step 4: CNF
# =====================================================
def eliminate_iff(s):
    op = s[0]
    if op == 'sym':
        return s
    if op == 'iff':
        a, b = eliminate_iff(s[1]), eliminate_iff(s[2])
        return And(Imp(a, b), Imp(b, a))
    return (op,) + tuple(eliminate_iff(x) for x in s[1:])


def eliminate_imp(s):
    op = s[0]
    if op == 'sym':
        return s
    if op == 'imp':
        return Or(Not(eliminate_imp(s[1])), eliminate_imp(s[2]))
    return (op,) + tuple(eliminate_imp(x) for x in s[1:])


def push_not(s):
    op = s[0]
    if op == 'sym':
        return s
    if op == 'not':
        inner = s[1]
        if inner[0] == 'sym':
            return s
        if inner[0] == 'not':                      # double negation
            return push_not(inner[1])
        if inner[0] == 'and':                      # De Morgan
            return Or(push_not(Not(inner[1])), push_not(Not(inner[2])))
        if inner[0] == 'or':
            return And(push_not(Not(inner[1])), push_not(Not(inner[2])))
    return (op,) + tuple(push_not(x) for x in s[1:])


def distribute(s):
    op = s[0]
    if op in ('sym', 'not'):
        return s
    a, b = distribute(s[1]), distribute(s[2])
    if op == 'and':
        return And(a, b)
    # op == 'or'
    if a[0] == 'and':
        return And(distribute(Or(a[1], b)), distribute(Or(a[2], b)))
    if b[0] == 'and':
        return And(distribute(Or(a, b[1])), distribute(Or(a, b[2])))
    return Or(a, b)


def to_cnf(s):
    return distribute(push_not(eliminate_imp(eliminate_iff(s))))


def _literals(s):
    """Collect literal strings from an OR-tree."""
    if s[0] == 'or':
        return _literals(s[1]) | _literals(s[2])
    if s[0] == 'not':
        return {'~' + s[1][1]}
    return {s[1]}


def _conjuncts(s):
    if s[0] == 'and':
        return _conjuncts(s[1]) + _conjuncts(s[2])
    return [s]


def negate_lit(l):
    return l[1:] if l.startswith('~') else '~' + l


def is_tautology(clause):
    return any(negate_lit(l) in clause for l in clause)


def to_clauses(s):
    clauses = set()
    for part in _conjuncts(to_cnf(s)):
        c = frozenset(_literals(part))
        if not is_tautology(c):
            clauses.add(c)
    return clauses


# =====================================================
# Step 5: resolution
# =====================================================
def resolve(ci, cj):
    result = []
    for lit in ci:
        neg = negate_lit(lit)
        if neg in cj:
            new = (ci - {lit}) | (cj - {neg})
            if not is_tautology(new):
                result.append(frozenset(new))
    return result


def resolution_unsat(clauses, limit=100_000):
    """True if the empty clause is derivable (clauses unsatisfiable),
    False if not, None if resolvent limit exceeded. Returns (result, count)."""
    clauses = set(clauses)
    count = 0
    while True:
        new = set()
        lst = list(clauses)
        for i in range(len(lst)):
            for j in range(i + 1, len(lst)):
                for r in resolve(lst[i], lst[j]):
                    count += 1
                    if count > limit:
                        return None, count
                    if len(r) == 0:
                        return True, count
                    new.add(r)
        if new <= clauses:
            return False, count
        clauses |= new


def pl_resolution(kb_clauses, alpha, limit=100_000):
    """KB |= alpha ?  Returns (result, resolvent count)."""
    neg_alpha = to_clauses(Not(alpha))
    return resolution_unsat(set(kb_clauses) | neg_alpha, limit)


# =====================================================
# Step 6: DPLL
# =====================================================
def lit_sym(l):
    return l[1:] if l.startswith('~') else l


def lit_val(l, model):
    s = lit_sym(l)
    if s not in model:
        return None
    return (not model[s]) if l.startswith('~') else model[s]


def find_pure_symbol(syms, clauses):
    for s in syms:
        pos = any(s in c for c in clauses)
        neg = any('~' + s in c for c in clauses)
        if pos != neg:
            return s, pos
    return None


def find_unit_clause(clauses, model):
    for c in clauses:
        unassigned = [l for l in c if lit_val(l, model) is None]
        if len(unassigned) == 1:
            l = unassigned[0]
            return lit_sym(l), not l.startswith('~')
    return None


def dpll(clauses, syms, model, ctr, use_unit=True, use_pure=True):
    ctr[0] += 1
    unknown = []
    for c in clauses:
        vals = [lit_val(l, model) for l in c]
        if True in vals:
            continue
        if all(v is False for v in vals):
            return False
        unknown.append(c)
    if not unknown:
        return model

    if use_pure:
        p = find_pure_symbol(syms, unknown)
        if p:
            s, v = p
            return dpll(clauses, [x for x in syms if x != s], {**model, s: v},
                        ctr, use_unit, use_pure)
    if use_unit:
        u = find_unit_clause(unknown, model)
        if u:
            s, v = u
            return dpll(clauses, [x for x in syms if x != s], {**model, s: v},
                        ctr, use_unit, use_pure)

    s = syms[0]
    rest = syms[1:]
    for v in (True, False):
        r = dpll(clauses, rest, {**model, s: v}, ctr, use_unit, use_pure)
        if r is not False:
            return r
    return False


def dpll_satisfiable(clauses, use_unit=True, use_pure=True):
    """Returns (model or False, number of dpll calls)."""
    clauses = list(clauses)
    syms = sorted({lit_sym(l) for c in clauses for l in c})
    ctr = [0]
    result = dpll(clauses, syms, {}, ctr, use_unit, use_pure)
    return result, ctr[0]


def dpll_entails(kb_clauses, alpha, **kw):
    clauses = set(kb_clauses) | to_clauses(Not(alpha))
    model, count = dpll_satisfiable(clauses, **kw)
    return model is False, count


# =====================================================
# Step 7: Wumpus
# =====================================================
def build_wumpus():
    P11, P12, P21, P22, P31 = (Sym(x) for x in ['P11', 'P12', 'P21', 'P22', 'P31'])
    B11, B21 = Sym('B11'), Sym('B21')
    rules = [
        Not(P11),
        Iff(B11, Or(P12, P21)),
        Iff(B21, Or(Or(P11, P22), P31)),
        Not(B11),
        B21,
    ]
    kb = rules[0]
    for r in rules[1:]:
        kb = And(kb, r)
    queries = {
        '~P12': Not(P12),
        '~P21': Not(P21),
        '~P22': Not(P22),
        '~P31': Not(P31),
        'P22|P31': Or(P22, P31),
    }
    return kb, queries


# =====================================================
# Step 8: random 3-SAT and comparison
# =====================================================
def random_3sat(n, m, seed):
    random.seed(seed)
    clauses = []
    for _ in range(m):
        vs = random.sample(range(1, n + 1), 3)
        clauses.append(frozenset(('~' if random.random() < 0.5 else '') + f"X{v}" for v in vs))
    return clauses


def tt_satisfiable(clauses, limit=200_000):
    syms = sorted({lit_sym(l) for c in clauses for l in c})
    count = 0
    for values in product([True, False], repeat=len(syms)):
        count += 1
        if count > limit:
            return None, count
        model = dict(zip(syms, values))
        if all(any(lit_val(l, model) for l in c) for c in clauses):
            return True, count
    return False, count


def fmt(count, result):
    return "хэт удаан" if result is None else str(count)





# =====================================================
# Checks and tables
# =====================================================
if __name__ == '__main__':
    P, Q, R = Sym('P'), Sym('Q'), Sym('R')
    A, B, C = Sym('A'), Sym('B'), Sym('C')

    print("=== Step 1-3 ===")
    s = Imp(And(P, Q), R)
    print(symbols(s), to_str(s))
    m1 = {'P': True, 'Q': False, 'R': True}
    print(pl_true(s, m1), pl_true(Iff(P, Q), m1), pl_true(Or(Not(P), Q), m1))  # True False False
    print(tt_entails(And(P, Imp(P, Q)), Q))            # (True, 4)
    print(tt_entails(Or(P, Q), P))                     # (False, ...)
    print(tt_entails(And(Not(Q), Imp(P, Q)), Not(P)))  # (True, 4)
    print(is_valid(Or(P, Not(P))), is_satisfiable(And(P, Not(P))))  # True False
    print_truth_table(Not(And(P, Q)))
    print_truth_table(Or(Not(P), Not(Q)))

    print("\n=== Step 4 ===")
    cl = to_clauses(Iff(B, Or(A, C)))
    print(len(cl), [sorted(c) for c in cl])            # 3 clauses
    tests = [Iff(P, Q), Imp(Or(P, Q), And(Q, R)), Not(Iff(P, Imp(Q, R))), Iff(Not(P), And(Q, Not(R)))]
    for t in tests:
        same = all(pl_true(t, m) == pl_true(to_cnf(t), m) for m in all_models(symbols(t)))
        print("to_cnf equivalent:", same)

    print("\n=== Step 5 ===")
    print(resolve(frozenset({'A', 'B'}), frozenset({'~A', 'C'})))   # [{B, C}]
    print(resolve(frozenset({'C'}), frozenset({'~C'})))             # [frozenset()]
    kb5 = [frozenset({'A', 'B'}), frozenset({'~A', 'C'}), frozenset({'~B', 'C'})]
    print(pl_resolution(kb5, C), pl_resolution(kb5, A))             # True / False

    print("\n=== Step 6 ===")
    c6 = [frozenset(x) for x in [{'A'}, {'~A', 'B'}, {'~B', 'C'}, {'~C', 'D', 'E'}, {'~D', '~E'}]]
    print(dpll_satisfiable(c6))
    c6u = [frozenset(x) for x in [{'A', 'B'}, {'~A', 'B'}, {'A', '~B'}, {'~A', '~B'}]]
    print(dpll_satisfiable(c6u))                                    # (False, ...)
    print(dpll_entails(kb5, C), dpll_entails(kb5, A))

    print("\n=== Step 7: Wumpus ===")
    kb, queries = build_wumpus()
    n_models = sum(1 for m in all_models(symbols(kb)) if pl_true(kb, m))
    print("KB models:", n_models)                                   # 3
    kb_cl = to_clauses(kb)
    for name, q in queries.items():
        r1 = tt_entails(kb, q)[0]
        r2 = pl_resolution(kb_cl, q)[0]
        r3 = dpll_entails(kb_cl, q)[0]
        print(f"KB |= {name}: tt={r1}, resolution={r2}, dpll={r3}")

    print("\n=== Step 8: comparison ===")
    rows = []
    for name in ['~P12', '~P22', 'P22|P31']:
        q = queries[name]
        t, tc = tt_entails(kb, q)
        r, rc = pl_resolution(kb_cl, q)
        _, dc = dpll_entails(kb_cl, q)
        rows.append((f"Wumpus: {name}", fmt(tc, t), fmt(rc, r), str(dc)))

    for n in (10, 15, 20):
        cls = random_3sat(n, 4 * n, seed=1)
        t, tc = tt_satisfiable(cls)
        r, rc = resolution_unsat(cls)
        _, dc = dpll_satisfiable(cls)
        rows.append((f"3-SAT, n = {n}", fmt(tc, t), fmt(rc, r), str(dc)))

    print(f"{'Бодлого':<20}{'Үнэний хүснэгт':<18}{'Резолюц':<14}{'DPLL':<8}")
    for row in rows:
        print(f"{row[0]:<20}{row[1]:<18}{row[2]:<14}{row[3]:<8}")

    print("\n=== Bonus: DPLL without unit / pure rules ===")
    for n in (10, 15, 20):
        cls = random_3sat(n, 4 * n, seed=1)
        full = dpll_satisfiable(cls)[1]
        no_unit = dpll_satisfiable(cls, use_unit=False)[1]
        no_pure = dpll_satisfiable(cls, use_pure=False)[1]
        neither = dpll_satisfiable(cls, use_unit=False, use_pure=False)[1]
        print(f"n={n}: full={full}, no unit={no_unit}, no pure={no_pure}, neither={neither}")



=== Step 1-3 ===
{'R', 'Q', 'P'} ((P & Q) => R)
True False False
(True, 4)
(False, 3)
(True, 4)
True False
P Q | ~(P & Q)
T T | F
T F | T
F T | T
F F | T
P Q | (~P | ~Q)
T T | F
T F | T
F T | T
F F | T

=== Step 4 ===
3 [['B', '~A'], ['A', 'C', '~B'], ['B', '~C']]
to_cnf equivalent: True
to_cnf equivalent: True
to_cnf equivalent: True
to_cnf equivalent: True

=== Step 5 ===
[frozenset({'B', 'C'})]
[frozenset()]
(True, 22) (False, 17)

=== Step 6 ===
({'A': True, 'B': True, 'C': True, 'D': True, 'E': False}, 6)
(False, 5)
(True, 4) (False, 4)

=== Step 7: Wumpus ===
KB models: 3
KB |= ~P12: tt=True, resolution=True, dpll=True
KB |= ~P21: tt=True, resolution=True, dpll=True
KB |= ~P22: tt=False, resolution=False, dpll=False
KB |= ~P31: tt=False, resolution=False, dpll=False
KB |= P22|P31: tt=True, resolution=True, dpll=True

=== Step 8: comparison ===
Бодлого             Үнэний хүснэгт    Резолюц       DPLL    
Wumpus: ~P12        128               9             7       
Wumpus: ~P22    